# 04 — Hybrid Retrieval

**What this notebook does:**
- Implements BM25 keyword search alongside vector semantic search
- Combines both scores with Reciprocal Rank Fusion (RRF)
- Compares pure vector vs hybrid retrieval on the same queries
- Shows why hybrid beats both individually

**Why hybrid?**
- Vector search is great for meaning/intent but misses exact terms
- BM25 is great for exact keywords but misses paraphrases
- Together they cover each other's weaknesses

In [1]:
import chromadb
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

client = chromadb.PersistentClient(path="../chroma_db")
collection = client.get_collection("my_docs")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print(f"Chunks loaded: {collection.count()}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Chunks loaded: 247


## Step 1 — Build the BM25 index
BM25 needs the full corpus loaded into memory as tokenised words.

In [2]:
# Load all stored chunks from ChromaDB
all_data = collection.get(include=["documents"])
all_chunks = all_data["documents"]
all_ids    = all_data["ids"]

print(f"Loaded {len(all_chunks)} chunks for BM25 index")

# Tokenise: lowercase and split by whitespace
# BM25 works on word tokens, not embeddings
tokenised = [chunk.lower().split() for chunk in all_chunks]

# Build BM25 index
bm25 = BM25Okapi(tokenised)

print(f"BM25 index built over {len(tokenised)} documents")
print(f"Vocabulary size: ~{len(set(w for doc in tokenised for w in doc)):,} unique words")

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Loaded 247 chunks for BM25 index
BM25 index built over 247 documents
Vocabulary size: ~3,431 unique words


## Step 2 — Individual retrieval functions

In [3]:
def vector_search(query, top_k=10):
    """Semantic vector search using ChromaDB."""
    vec = embed_model.encode([query]).tolist()
    res = collection.query(query_embeddings=vec, n_results=top_k)
    return res["ids"][0], res["distances"][0], res["documents"][0]


def bm25_search(query, top_k=10):
    """Keyword BM25 search."""
    query_tokens = query.lower().split()
    scores = bm25.get_scores(query_tokens)

    # Get top_k indices sorted by score descending
    top_indices = np.argsort(scores)[::-1][:top_k]

    top_ids    = [all_ids[i] for i in top_indices]
    top_scores = [scores[i] for i in top_indices]
    top_chunks = [all_chunks[i] for i in top_indices]

    return top_ids, top_scores, top_chunks

## Step 3 — Reciprocal Rank Fusion (RRF)
RRF combines two ranked lists into one without needing to normalise scores.
Formula: `score(chunk) = 1/(k + rank_in_list1) + 1/(k + rank_in_list2)`

In [4]:
def reciprocal_rank_fusion(vector_ids, bm25_ids, k=60):
    """
    Combines two ranked lists using Reciprocal Rank Fusion.
    k=60 is standard — higher k reduces the impact of top ranks.
    """
    scores = {}

    for rank, cid in enumerate(vector_ids):
        scores[cid] = scores.get(cid, 0) + 1 / (k + rank + 1)

    for rank, cid in enumerate(bm25_ids):
        scores[cid] = scores.get(cid, 0) + 1 / (k + rank + 1)

    # Sort by combined score descending
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return ranked


def hybrid_search(query, top_k=3):
    """
    Full hybrid search: BM25 + Vector → RRF → top_k results.
    """
    # Get top 10 from each method
    v_ids, v_dists, v_chunks = vector_search(query, top_k=10)
    b_ids, b_scores, b_chunks = bm25_search(query, top_k=10)

    # Fuse rankings
    fused = reciprocal_rank_fusion(v_ids, b_ids)

    # Get top_k chunk texts from fused ranking
    results = []
    for cid, rrf_score in fused[:top_k]:
        # Find the chunk text
        if cid in v_ids:
            idx = v_ids.index(cid)
            chunk_text = v_chunks[idx]
        else:
            idx = b_ids.index(cid)
            chunk_text = b_chunks[idx]

        in_vector = cid in v_ids[:3]
        in_bm25   = cid in b_ids[:3]
        source    = []
        if in_vector: source.append("vector")
        if in_bm25:   source.append("bm25")

        results.append({
            "id": cid,
            "rrf_score": rrf_score,
            "text": chunk_text,
            "found_by": " + ".join(source) if source else "rrf_only"
        })

    return results

## Step 4 — Compare vector vs hybrid on same queries

In [5]:
def compare_methods(query):
    print(f"\n{'='*60}")
    print(f" Query: {query}")
    print(f"{'='*60}")

    # Vector only
    v_ids, v_dists, _ = vector_search(query, top_k=3)
    print(f"\nVector search top-3:")
    for cid, dist in zip(v_ids, v_dists):
        print(f"  {cid}  dist={dist:.4f}")

    # BM25 only
    b_ids, b_scores, _ = bm25_search(query, top_k=3)
    print(f"\nBM25 search top-3:")
    for cid, score in zip(b_ids, b_scores):
        print(f"  {cid}  score={score:.4f}")

    # Hybrid
    hybrid = hybrid_search(query, top_k=3)
    print(f"\nHybrid (RRF) top-3:")
    for r in hybrid:
        print(f"  {r['id']}  rrf={r['rrf_score']:.4f}  found_by={r['found_by']}")
        print(f"  {r['text'][:150]}...")


# CHANGE these queries to match your document
compare_methods("What is the main topic of this document?")
compare_methods("specific technical term from your PDF")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



 Query: What is the main topic of this document?

Vector search top-3:
  chunk_160  dist=0.6072
  chunk_195  dist=0.6648
  chunk_173  dist=0.6840

BM25 search top-3:
  chunk_113  score=11.0506
  chunk_220  score=10.1863
  chunk_149  score=9.5665

Hybrid (RRF) top-3:
  chunk_244  rrf=0.0288  found_by=rrf_only
  [Page 86]
  
UNIT 3: PROCESS SYNCHRONIZATION   
 
 
3. How many and what type of resources is the process holding. 
 
4. How many more resources does ...
  chunk_160  rrf=0.0164  found_by=vector
  [Page 56]
 
 
 1      n . 
burst...
  chunk_113  rrf=0.0164  found_by=bm25
  The sequential processes can be programmed individually without having to worry 
about other processes running on the processor. That is, layer 0 prov...

 Query: specific technical term from your PDF

Vector search top-3:
  chunk_167  dist=0.6368
  chunk_134  dist=0.6738
  chunk_166  dist=0.7089

BM25 search top-3:
  chunk_76  score=6.0358
  chunk_2  score=5.0721
  chunk_62  score=4.8497

Hybrid (RRF) to

## Step 5 — Drop-in replacement for notebook 03
This replaces `retrieve_chunks()` — copy this into notebook 03 to upgrade it.

In [6]:
def retrieve_chunks_hybrid(query, top_k=3):
    """
    Drop-in replacement for retrieve_chunks() in notebook 03.
    Returns same format: (chunks, ids, scores)
    """
    results = hybrid_search(query, top_k=top_k)
    chunks = [r["text"] for r in results]
    ids    = [r["id"]   for r in results]
    scores = [r["rrf_score"] for r in results]
    return chunks, ids, scores

# Test it
chunks, ids, scores = retrieve_chunks_hybrid("What is this document about?")
print("Hybrid retrieval working:")
for cid, score in zip(ids, scores):
    print(f"  {cid}  rrf_score={score:.4f}")

Hybrid retrieval working:
  chunk_160  rrf_score=0.0164
  chunk_220  rrf_score=0.0164
  chunk_134  rrf_score=0.0161


## Key observations
- Chunks marked `found_by=vector + bm25` are the strongest - both methods agree
- Chunks only found by `bm25` are likely exact keyword matches - very reliable
- Chunks only found by `vector` are likely semantic matches - good for paraphrases
- RRF score is not a distance - higher is better

**Next:** `05_reranker.ipynb` 